# Scrapper de Metro Cuadrado

Para la obtención de datos del proyecto Machine Learning Techniques

In [1]:
!pip3 install requests beautifulsoup4 lxml --quiet

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [2]:
# librerías
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
from tqdm import tqdm

In [5]:
# variables generales - por defecto son de arriendo
city = "bogota"
property_type = "apartaestudio-apartamento-casa"
num_pages = 1

# URL base
BASE_URL = "https://www.metrocuadrado.com"

# listado de 
LISTING_URL = f"https://www.metrocuadrado.com/{property_type}/arriendo/{city}/"

# headers para simular un navegador
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

In [36]:
def get_listing_urls(max_pages = 10):
    """
    Obtener las URLs de los listados de propiedades desde Metro Cuadrado con las configuraciones dadas (ciudad, tipo de propiedad, número de páginas).
    """

    urls = []
    for page in tqdm(range(1, max_pages + 1), desc="Obteniendo URLs de listados"):
        # pagina actual
        url = f"{LISTING_URL}?page={page}"

        # solicitud get
        response = requests.get(url, headers=HEADERS, timeout=30)

        if response.status_code != 200:
            print(f"Error al acceder a la página {page}: {response.status_code}")
            continue

        # scrappear el contenido
        soup = BeautifulSoup(response.content, "lxml")
        print(soup.prettify())
        cards = soup.find_all("div", class_="property-card__content")
        
        # extraer url dentro de cards
        for card in cards:
            a = card.find("a", href=True)
            if a:
                href = a["href"]
                if href.startswith("/inmueble/"):
                    clean_href = href.split("?")[0]
                    full_url = BASE_URL + clean_href
                    urls.append(full_url)


        break


In [52]:
import pandas as pd
import time
import re
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

def extraer_minucia_final(driver, url):
    driver.get(url)
    time.sleep(6) # Espera para que el Shadow DOM se hidrate
    
    html_source = driver.page_source
    
    # Función auxiliar para buscar texto con Regex
    def buscar(pattern, texto):
        match = re.search(pattern, texto)
        return match.group(1).strip() if match else "N/A"

    data = {
        "url": url,
        "precio_arriendo": buscar(r'text-\[24px\].*?>\$ ([\d\.]+)</div>', html_source),
        "precio_administracion": buscar(r'Administración:\s*\$\s*([\d\.]+)', html_source),
        "descripcion": "N/A",
        "baños": buscar(r'>(\d+)\s*bañ\.', html_source),
        "habitaciones": buscar(r'>(\d+)\s*hab\.', html_source),
        "caracteristicas_interiores": [],
        "caracteristicas_exteriores": [],
        "latitud": "N/A",
        "longitud": "N/A"
    }

    try:
        # 1. Extraer el comentario (Acerca de este inmueble)
        # Buscamos el div que sigue al h2 de "Acerca de este inmueble"
        try:
            desc_element = driver.find_element(By.XPATH, "//h2[contains(text(), 'Acerca de este')]/following-sibling::div")
            data["descripcion"] = desc_element.text
        except: pass

        # 2. Extraer Características (Interiores y Exteriores)
        # Estas están en pt-tag con element-id
        tags = driver.find_elements(By.TAG_NAME, "pt-tag")
        for tag in tags:
            tag_id = tag.get_attribute("element-id")
            if tag_id:
                # Si el tag está dentro del acordeón de Interiores o Exteriores
                # Por ahora los guardamos todos y filtramos por contexto
                if tag.find_element(By.XPATH, "./..").get_attribute("class") == "flex flex-wrap gap-2":
                    data["caracteristicas_interiores"].append(tag_id)

        # 3. Ubicación (Coordenadas)
        try:
            map_tag = driver.find_element(By.TAG_NAME, "pt-marker-map")
            data["latitud"] = map_tag.get_attribute("latitude")
            data["longitud"] = map_tag.get_attribute("longitude")
        except: pass

    except Exception as e:
        print(f"Error en {url}: {e}")

    # Limpiar las listas para que se vean bien en el DataFrame
    data["caracteristicas_interiores"] = ", ".join(data["caracteristicas_interiores"])
    
    return data

def ejecutar_scraping_pro(ciudad):
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    url_resultados = f"https://www.metrocuadrado.com/apartamento-apartaestudio/arriendo/{ciudad.lower()}/"
    
    try:
        driver.get(url_resultados)
        time.sleep(5)
        
        # Recolectar links de inmuebles
        links = list(set([a.get_attribute('href') for a in driver.find_elements(By.TAG_NAME, "a") 
                         if a.get_attribute('href') and '/inmueble/' in a.get_attribute('href')]))
        
        print(f"Iniciando extracción de {len(links[:5])} propiedades...")
        
        final_list = []
        for link in links[:5]: # Probar con 5 primero
            print(f"Extrayendo: {link}")
            final_list.append(extraer_minucia_final(driver, link))
            
        return pd.DataFrame(final_list)
        
    finally:
        driver.quit()

# --- RUN ---
df = ejecutar_scraping_pro("bogota")
df

Iniciando extracción de 5 propiedades...
Extrayendo: https://www.metrocuadrado.com/inmueble/arriendo-apartaestudio-bogota-cedro-narvaez-1-habitaciones-1-banos-2-garajes/9859-M6367273?src_url=%2Fapartamento-apartaestudio%2Farriendo%2Fbogota%2F
Extrayendo: https://www.metrocuadrado.com/inmueble/arriendo-apartamento-bogota-santa-barbara-occidental-1-habitaciones-1-banos-1-garajes/MC6245749?src_url=%2Fapartamento-apartaestudio%2Farriendo%2Fbogota%2F
Extrayendo: https://www.metrocuadrado.com/inmueble/arriendo-apartamento-bogota-chico-1-habitaciones-2-banos-2-garajes/10162-M6331418?src_url=%2Fapartamento-apartaestudio%2Farriendo%2Fbogota%2F
Extrayendo: https://www.metrocuadrado.com/inmueble/arriendo-apartaestudio-bogota-salitre-1-habitaciones-1-banos/897-M6296494?src_url=%2Fapartamento-apartaestudio%2Farriendo%2Fbogota%2F
Extrayendo: https://www.metrocuadrado.com/inmueble/arriendo-apartamento-bogota-la-veracruz-1-habitaciones-2-banos/MC6421039?src_url=%2Fapartamento-apartaestudio%2Farriendo%

,url,precio_arriendo,precio_administracion,descripcion,baños,habitaciones,caracteristicas_interiores,caracteristicas_exteriores,latitud,longitud
0,https://www.metrocuadrado.com/inmueble/arriend...,3.200.000,400.000,DISPONIBLE PARTIR DEL 01 DE ABRIL 2026. Aparta...,N/A,N/A,"Apto para niños, Alarma, Con chimenea, Citófon...",[],N/A,N/A
1,https://www.metrocuadrado.com/inmueble/arriend...,2.100.000,517.000,Este apartamento es de esos que se sienten bie...,N/A,N/A,"Con chimenea, Citófonos, Cocina integral, Piso...",[],N/A,N/A
2,https://www.metrocuadrado.com/inmueble/arriend...,3.950.000,N/A,"Arriendo apartamento, excelente vista, área co...",N/A,N/A,"Zona de lavanderia, Número de piso 7, Vista ex...",[],N/A,N/A
3,https://www.metrocuadrado.com/inmueble/arriend...,2.956.000,N/A,"Descubre este hermoso Apartaestudio Amoblado, ...",N/A,N/A,"Citófonos, Tipo de calentador gas, Tipo de Coc...",[],N/A,N/A
4,https://www.metrocuadrado.com/inmueble/arriend...,2.900.000,N/A,Apartamento ubicado en el emblemático Edificio...,N/A,N/A,"Baño Auxiliar, Baño de Servicio, Citófonos, Co...",[],N/A,N/A


In [53]:
import pandas as pd
import time
import re
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

def extraer_info_completa(driver, url):
    driver.get(url)
    time.sleep(7) # Espera para que carguen los componentes pt-
    
    # Obtenemos el HTML renderizado
    html_source = driver.page_source
    
    def buscar_con_regex(patron, texto):
        match = re.search(patron, texto)
        return match.group(1).strip() if match else "N/A"

    # 1. Extraer Características Básicas (Regex sobre el HTML cargado)
    # Buscamos patrones como "1 hab.", "2 bañ.", "1 par."
    info = {
        "url": url,
        "precio_arriendo": buscar_con_regex(r'text-\[24px\].*?>\$ ([\d\.]+)</div>', html_source),
        "precio_administracion": buscar_con_regex(r'Administración:\s*\$\s*([\d\.]+)', html_source),
        "habitaciones": buscar_con_regex(r'>(\d+)\s*hab\.', html_source),
        "baños": buscar_con_regex(r'>(\d+)\s*bañ\.', html_source),
        "parqueaderos": buscar_con_regex(r'>(\d+)\s*par\.', html_source),
        "descripcion": "N/A",
        "caracteristicas": [],
        "latitud": "N/A",
        "longitud": "N/A"
    }

    try:
        # 2. Descripción "Acerca de este inmueble"
        # Buscamos el div que contiene el texto largo de la descripción
        try:
            desc_xpath = "//h2[contains(text(), 'Acerca de este')]/following-sibling::div"
            info["descripcion"] = driver.find_element(By.XPATH, desc_xpath).text
        except: pass

        # 3. Características del inmueble (Interiores / Zonas Comunes)
        # Extraemos el element-id de cada pt-tag que son las etiquetas de características
        tags = driver.find_elements(By.TAG_NAME, "pt-tag")
        for tag in tags:
            tag_id = tag.get_attribute("element-id")
            if tag_id and tag_id not in ["propertyState"]: # Ignorar el tag de 'Usado'
                info["caracteristicas"].append(tag_id)
        
        info["caracteristicas"] = ", ".join(list(set(info["caracteristicas"])))

        # 4. Latitud y Longitud (Atributos del componente pt-marker-map)
        try:
            map_elem = driver.find_element(By.TAG_NAME, "pt-marker-map")
            info["latitud"] = map_elem.get_attribute("latitude")
            info["longitud"] = map_elem.get_attribute("longitude")
        except:
            # Plan B: Buscar coordenadas en el script de configuración si el tag no está
            coords_match = re.search(r'"latitude":\s*"([\d\.-]+)".*?"longitude":\s*"([\d\.-]+)"', html_source)
            if coords_match:
                info["latitud"], info["longitud"] = coords_match.groups()

    except Exception as e:
        print(f"Error procesando {url}: {e}")

    return info

def scrap_ciudad(ciudad):
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    url_base = f"https://www.metrocuadrado.com/apartamento-apartaestudio/arriendo/{ciudad.lower()}/"
    
    try:
        driver.get(url_base)
        time.sleep(6)
        
        # Recolectar links (filtramos para que solo sean de inmuebles)
        links = [a.get_attribute('href') for a in driver.find_elements(By.TAG_NAME, "a") 
                 if a.get_attribute('href') and '/inmueble/' in a.get_attribute('href')]
        links = list(set(links))
        
        print(f"Se encontraron {len(links)} propiedades en {ciudad}. Iniciando minucia...")
        
        final_data = []
        for l in links[:5]: # Cambia este número para traer más
            print(f"Extrayendo: {l}")
            final_data.append(extraer_info_completa(driver, l))
            
        return pd.DataFrame(final_data)
        
    finally:
        driver.quit()

# EJECUCIÓN
df_final = scrap_ciudad("bogota")
df_final

Se encontraron 6 propiedades en bogota. Iniciando minucia...
Extrayendo: https://www.metrocuadrado.com/inmueble/arriendo-apartamento-bogota-santa-barbara-central-1-habitaciones-1-banos-1-garajes/696-M6328523?src_url=%2Fapartamento-apartaestudio%2Farriendo%2Fbogota%2F
Extrayendo: https://www.metrocuadrado.com/inmueble/arriendo-apartamento-bogota-santa-barbara-occidental-1-habitaciones-1-banos-1-garajes/MC6245749?src_url=%2Fapartamento-apartaestudio%2Farriendo%2Fbogota%2F
Extrayendo: https://www.metrocuadrado.com/inmueble/arriendo-apartamento-bogota-chico-norte-1-habitaciones-1-banos/21282-M6099315?src_url=%2Fapartamento-apartaestudio%2Farriendo%2Fbogota%2F
Extrayendo: https://www.metrocuadrado.com/inmueble/arriendo-apartamento-bogota-la-veracruz-1-habitaciones-2-banos/MC6421039?src_url=%2Fapartamento-apartaestudio%2Farriendo%2Fbogota%2F
Extrayendo: https://www.metrocuadrado.com/inmueble/arriendo-apartamento-bogota-san-patricio-1-habitaciones-2-banos-1-garajes/MC6415207?src_url=%2Faparta

,url,precio_arriendo,precio_administracion,habitaciones,baños,parqueaderos,descripcion,caracteristicas,latitud,longitud
0,https://www.metrocuadrado.com/inmueble/arriend...,2.400.000,N/A,N/A,N/A,N/A,"Apartamento en arriendo en Santa Barbara, ubic...","Terraza/Balcón ninguno, Depósito 1, Tipo de es...",N/A,N/A
1,https://www.metrocuadrado.com/inmueble/arriend...,2.100.000,517.000,N/A,N/A,N/A,Este apartamento es de esos que se sienten bie...,"Citófonos, Shut de basura, imageCounter, Ubica...",N/A,N/A
2,https://www.metrocuadrado.com/inmueble/arriend...,3.550.000,726.000,N/A,N/A,N/A,"Ubicado en el exclusivo sector del Chicó, sobr...","Tipo de estufa electrica, Sobre vía principal,...",N/A,N/A
3,https://www.metrocuadrado.com/inmueble/arriend...,2.900.000,N/A,N/A,N/A,N/A,Apartamento ubicado en el emblemático Edificio...,"Baño de Servicio, Citófonos, Terraza Rooftop, ...",N/A,N/A
4,https://www.metrocuadrado.com/inmueble/arriend...,4.685.000,305.000,N/A,N/A,N/A,"Arriendo apartamento nuevo en San Patricio, ex...","Citófonos, Terraza Rooftop, Cerca Supermercado...",N/A,N/A


In [54]:
import pandas as pd
import time
import re
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

def extraer_todo_inmueble(driver, url):
    driver.get(url)
    time.sleep(8) # Tiempo suficiente para que cargue el mapa y los componentes pt-
    
    html_source = driver.page_source
    
    def limpiar_numero(texto):
        if texto == "N/A": return texto
        return re.sub(r'[^\d]', '', texto)

    # Inicializamos el diccionario con todo lo que pides
    res = {
        "url": url,
        "titulo": "N/A",
        "precio_arriendo": "N/A",
        "precio_administracion": "N/A",
        "habitaciones": "N/A",
        "baños": "N/A",
        "parqueaderos": "N/A",
        "area_construida": "N/A",
        "estrato": "N/A",
        "estado": "N/A",
        "latitud": "N/A",
        "longitud": "N/A",
        "descripcion": "N/A",
        "caracteristicas": []
    }

    try:
        # --- 1. PRECIOS (Usando las clases que me pasaste) ---
        res["precio_arriendo"] = limpiar_numero(re.search(r'text-\[24px\].*?>\$ ([\d\.]+)</div>', html_source).group(1) if "$" in html_source else "N/A")
        admin_match = re.search(r'Administración:\s*\$\s*([\d\.]+)', html_source)
        if admin_match: res["precio_administracion"] = limpiar_numero(admin_match.group(1))

        # --- 2. HAB / BAÑOS / PARQUEADEROS (Búsqueda por texto directo en pantalla) ---
        # Buscamos en el HTML renderizado los patrones que me diste
        res["habitaciones"] = re.search(r'(\d+)\s*hab\.', html_source).group(1) if "hab." in html_source else "N/A"
        res["baños"] = re.search(r'(\d+)\s*bañ\.', html_source).group(1) if "bañ." in html_source else "N/A"
        res["parqueaderos"] = re.search(r'(\d+)\s*par\.', html_source).group(1) if "par." in html_source else "N/A"

        # --- 3. DATOS DE ESTRATO Y ÁREA ---
        res["area_construida"] = re.search(r'Área Construida\s*([\d\w²\s]+)', html_source).group(1) if "Área Construida" in html_source else "N/A"
        res["estrato"] = re.search(r'Estrato\s*(\d+)', html_source).group(1) if "Estrato" in html_source else "N/A"

        # --- 4. COORDENADAS (Directo del componente pt-marker-map) ---
        try:
            map_tag = driver.find_element(By.TAG_NAME, "pt-marker-map")
            res["latitud"] = map_tag.get_attribute("latitude")
            res["longitud"] = map_tag.get_attribute("longitude")
        except:
            # Si el tag no responde, buscamos en el JSON interno de la página
            coords = re.search(r'"latitude":"([\d\.-]+)","longitude":"([\d\.-]+)"', html_source)
            if coords:
                res["latitud"], res["longitud"] = coords.groups()

        # --- 5. DESCRIPCIÓN ---
        try:
            res["descripcion"] = driver.find_element(By.XPATH, "//h2[contains(text(), 'Acerca de este')]/following-sibling::div").text
        except: pass

        # --- 6. CARACTERÍSTICAS (Extraer de los pt-tag) ---
        tags = driver.find_elements(By.TAG_NAME, "pt-tag")
        for t in tags:
            id_val = t.get_attribute("element-id")
            if id_val and id_val not in ["propertyState", "btnSeeMore"]:
                res["caracteristicas"].append(id_val)
        
        # --- 7. TÍTULO Y ESTADO ---
        try: res["titulo"] = driver.find_element(By.TAG_NAME, "h1").text
        except: pass
        try: res["estado"] = driver.find_element(By.CSS_SELECTOR, "pt-tag[element-id='propertyState']").text
        except: pass

    except Exception as e:
        print(f"Error parcial en {url}: {e}")

    res["caracteristicas"] = ", ".join(list(set(res["caracteristicas"])))
    return res

def iniciar_super_scraping(ciudad, limite=5):
    options = Options()
    options.add_argument("--start-maximized")
    # User agent para evitar bloqueos
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    url_base = f"https://www.metrocuadrado.com/apartamento-apartaestudio/arriendo/{ciudad.lower()}/"
    
    try:
        driver.get(url_base)
        time.sleep(6)
        
        # Capturamos los links de los inmuebles
        links = list(set([a.get_attribute('href') for a in driver.find_elements(By.TAG_NAME, "a") 
                         if a.get_attribute('href') and '/inmueble/' in a.get_attribute('href')]))
        
        print(f"Encontrados {len(links)} inmuebles. Procesando los primeros {limite}...")
        
        data_final = []
        for l in links[:limite]:
            print(f"Analizando: {l}")
            data_final.append(extraer_todo_inmueble(driver, l))
            
        return pd.DataFrame(data_final)
    finally:
        driver.quit()

# EJECUCIÓN
df = iniciar_super_scraping("bogota", limite=3)
df

Encontrados 6 inmuebles. Procesando los primeros 3...
Analizando: https://www.metrocuadrado.com/inmueble/arriendo-apartamento-bogota-los-cedritos-1-habitaciones-1-banos-1-garajes/MC2748942?src_url=%2Fapartamento-apartaestudio%2Farriendo%2Fbogota%2F
Analizando: https://www.metrocuadrado.com/inmueble/arriendo-apartaestudio-bogota-santa-barbara-alta-1-habitaciones-1-banos/897-M6319177?src_url=%2Fapartamento-apartaestudio%2Farriendo%2Fbogota%2F
Analizando: https://www.metrocuadrado.com/inmueble/arriendo-apartamento-bogota-la-veracruz-1-habitaciones-2-banos/MC6421039?src_url=%2Fapartamento-apartaestudio%2Farriendo%2Fbogota%2F


,url,titulo,precio_arriendo,precio_administracion,habitaciones,baños,parqueaderos,area_construida,estrato,estado,latitud,longitud,descripcion,caracteristicas
0,https://www.metrocuadrado.com/inmueble/arriend...,"Apartamento En Arriendo, Cedritos",1600000,357050,N/A,N/A,N/A,54m²,4,Usado,N/A,N/A,"Códomo apartamento, Muy iluminado. Una alcoba ...","propertyType, Parqueadero visitantes, Cerca Pa..."
1,https://www.metrocuadrado.com/inmueble/arriend...,"Apartaestudio En Arriendo, Santa Barbara Alta",1900000,N/A,N/A,N/A,N/A,22,6,Usado,N/A,N/A,"En el corazón de la ciudad, se encuentra este ...","propertyType, Sobre vía principal, Cerca Super..."
2,https://www.metrocuadrado.com/inmueble/arriend...,"Apartamento En Arriendo, CENTRO LAS AGUAS Cent...",2900000,N/A,N/A,N/A,N/A,54,3,Usado,N/A,N/A,Apartamento ubicado en el emblemático Edificio...,"Baño de Servicio, Citófonos, Terraza Rooftop, ..."


In [55]:
import requests
import pandas as pd
import time

def scraping_metrocuadrado_total(ciudad="bogotá", negocio="arriendo", total_resultados=100):
    # La llave que encontraste (llave maestra pública)
    api_key = 'P1MfFHfQMOtL16Zpg36NcntJYCLFm8FqFfudnavl'
    
    headers = {
        'X-Api-Key': api_key,
        'Content-Type': 'application/json',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
    }

    inmuebles_lista = []
    # La API permite pedir máximo 50 por paquete
    paso = 50 

    print(f"--- Iniciando extracción de {negocio} en {ciudad} ---")

    for desde in range(0, total_resultados, paso):
        # URL de la API Rest (Trae los datos puros sin pasar por el navegador)
        url = (f"https://www.metrocuadrado.com/rest-search/search?"
               f"realEstateTypeList=apartamento&"
               f"realEstateBusinessList={negocio}&"
               f"city={ciudad.lower()}&"
               f"from={desde}&"
               f"size={paso}")

        try:
            response = requests.get(url, headers=headers)
            
            if response.status_code != 200:
                print(f"Error {response.status_code}. Es posible que la llave haya expirado.")
                break

            data = response.json()
            items = data.get('results', [])

            if not items:
                print("No hay más resultados.")
                break

            for i in items:
                # Extraemos absolutamente todos los campos disponibles
                registro = {
                    'ID': i.get('midinmueble'),
                    'Titulo': i.get('mtitle'),
                    'Precio': i.get('mvalorarriendo') if negocio == "arriendo" else i.get('mvalorventa'),
                    'Administracion': i.get('mvaloradministracion', 0),
                    'Habitaciones': i.get('mnrohabitaciones'),
                    'Baños': i.get('mnrobanos'),
                    'Parqueaderos': i.get('mnrogarajes'),
                    'Area_m2': i.get('marea'),
                    'Estrato': i.get('mestrato'),
                    'Barrio': i.get('mnombrecomunbarrio'),
                    'Ciudad': i.get('mciudad'),
                    'Latitud': i.get('mlatitud'),  # ¡Aquí están las coordenadas!
                    'Longitud': i.get('mlongitud'),
                    'Estado': i.get('mestadoinmueble'),
                    'Antiguedad': i.get('mantiguedad'),
                    'Descripcion': i.get('mdescripcion'), # El comentario "Acerca de este inmueble"
                    'Link': "https://www.metrocuadrado.com" + i.get('link', ''),
                    'Caracteristicas_Generales': ", ".join(i.get('mcaracteristicas_generales', [])),
                    'Caracteristicas_Interiores': ", ".join(i.get('mcaracteristicas_interiores', [])),
                    'Caracteristicas_Exteriores': ", ".join(i.get('mcaracteristicas_exteriores', []))
                }
                inmuebles_lista.append(registro)

            print(f"Paquete completado: {len(inmuebles_lista)} inmuebles procesados.")
            time.sleep(1) # Un segundo de respiro para no saturar

        except Exception as e:
            print(f"Ocurrió un error inesperado: {e}")
            break

    # Convertir a DataFrame de Pandas
    df = pd.DataFrame(inmuebles_lista)
    return df

# --- EJECUCIÓN DEL PROGRAMA ---
# Puedes subir el número en 'total_resultados' para traer miles de datos
df_final = scraping_metrocuadrado_total(ciudad="bogota", negocio="arriendo", total_resultados=5)
df_final

--- Iniciando extracción de arriendo en bogota ---
Paquete completado: 65 inmuebles procesados.


,ID,Titulo,Precio,Administracion,Habitaciones,Baños,Parqueaderos,Area_m2,Estrato,Barrio,Ciudad,Latitud,Longitud,Estado,Antiguedad,Descripcion,Link,Caracteristicas_Generales,Caracteristicas_Interiores,Caracteristicas_Exteriores
0,MC2748942,None,1600000,0,None,1,1,54.0,None,Cedritos,"{'id': '1', 'nombre': 'Bogotá D.C.'}",None,None,Usado,None,None,https://www.metrocuadrado.com/inmueble/arriend...,,,
1,MC6317896,None,1350000,0,None,1,0,40.0,None,LOS MONJES Engativá,"{'id': '1', 'nombre': 'Bogotá D.C.'}",None,None,Usado,None,None,https://www.metrocuadrado.com/inmueble/arriend...,,,
2,MC6332907,None,1400000,0,None,1,0,60.0,None,REMANSO SUR Puente Aranda,"{'id': '1', 'nombre': 'Bogotá D.C.'}",None,None,Usado,None,None,https://www.metrocuadrado.com/inmueble/arriend...,,,
3,MC5440684,None,2500000,0,None,2,1,60.0,None,ESTACION DE LA SABANA Puente Aranda,"{'id': '1', 'nombre': 'Bogotá D.C.'}",None,None,Usado,None,None,https://www.metrocuadrado.com/inmueble/arriend...,,,
4,MC2769920,None,1450000,0,None,1,0,60.0,None,CIUDAD MONTES Puente Aranda,"{'id': '1', 'nombre': 'Bogotá D.C.'}",None,None,Usado,None,None,https://www.metrocuadrado.com/inmueble/arriend...,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,9951-M6004861,None,13000000,0,None,4,2,191.0,None,LOS ROSALES,"{'id': '1', 'nombre': 'Bogotá D.C.'}",None,None,Usado,None,None,https://www.metrocuadrado.com/inmueble/arriend...,,,
61,MC6303471,None,5800000,0,None,2,2,100.0,None,SANTA BARBARA CENTRAL Santa Bárbara,"{'id': '1', 'nombre': 'Bogotá D.C.'}",None,None,Usado,None,None,https://www.metrocuadrado.com/inmueble/arriend...,,,
62,MC6411937,None,2000000,0,None,1,0,36.0,None,COLINA EL PLAN Colina y Alrededores,"{'id': '1', 'nombre': 'Bogotá D.C.'}",None,None,Usado,None,None,https://www.metrocuadrado.com/inmueble/arriend...,,,
63,MC6398809,None,2400000,0,None,1,1,34.0,None,LA COLINA CAMPESTRE Colina y Alrededores,"{'id': '1', 'nombre': 'Bogotá D.C.'}",None,None,Usado,None,None,https://www.metrocuadrado.com/inmueble/arriend...,,,


In [ ]:
import requests
import pandas as pd
import time

def scraping_total_metrocuadrado(ciudad="bogotá", negocio="arriendo", total_pedir=200):
    # La llave que confirmaste
    api_key = 'P1MfFHfQMOtL16Zpg36NcntJYCLFm8FqFfudnavl'
    
    headers = {
        'Content-type': 'application/json',
        'X-Api-Key': api_key,
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    inmuebles = []
    size = 50 # Paquetes de 50 en 50

    for j in range(0, int(total_pedir/size)):
        offset = j * size
        url = f"https://www.metrocuadrado.com/rest-search/search?realEstateTypeList=apartamento&realEstateBusinessList={negocio}&city={ciudad.lower()}&from={offset}&size={size}"
        
        try:
            r = requests.get(url, headers=headers)
            if r.status_code != 200:
                print(f"Error en la petición: {r.status_code}")
                break
                
            data = r.json()
            
            for i in data['results']:
                print(i)
                # Mapeo exacto según el JSON de MetroCuadrado
                inmuebles.append({
                    'ID': i.get('midinmueble'),
                    'Link': "https://www.metrocuadrado.com" + i.get('link', ''),
                    'Titulo': i.get('title'),
                    'Valor_Arriendo': i.get('mvalorarriendo'),
                    'Administracion': i.get('data', {}).get('mvaloradministracion'),
                    'AreaConstruida': i.get('mareac'),
                    'AreaHabitable': i.get('marea'),
                    'Habitaciones': i.get('mnrocuartos'),
                    'Baños': i.get('mnrobanos'),
                    'Parqueaderos': i.get('mnrogarajes'),
                    'Estrato': i.get('mestrato'),
                    'Barrio': i.get('mbarrio'),
                    'Barrio_Comun': i.get('mnombrecomunbarrio'),
                    'Latitud': i.get('localizacion', {}).get('lat'),
                    'Longitud': i.get('localizacion', {}).get('lon'),
                    'Antiguedad': i.get('mantiguedad'),
                    'Descripcion': i.get('mdescripcion'),
                    'Carac_Generales': ", ".join(i.get('mcaracteristicas_generales', [])),
                    'Carac_Interiores': ", ".join(i.get('mcaracteristicas_interiores', [])),
                    'Carac_Exteriores': ", ".join(i.get('mcaracteristicas_exteriores', []))
                })
            
            print(f"Descargados {len(inmuebles)} inmuebles...")
            time.sleep(0.5) # Evitar bloqueos

        except Exception as e:
            print(f"Error procesando datos: {e}")
            break

    return pd.DataFrame(inmuebles)

# --- EJECUCIÓN ---
# Sacamos 300 apartamentos en Bogotá
df = scraping_total_metrocuadrado(ciudad="bogota", negocio="arriendo", total_pedir=50)
df

{'contactPhone': '3013095509', 'whatsapp': '573013095509', 'title': 'Apartamento en Arriendo, CHICO NORTE   Zona Urbana, Bogotá D.C.', 'link': '/inmueble/arriendo-apartamento-bogota-cr.-green-house-1-habitaciones-2-banos-1-garajes/MC6287274', 'imageLink': 'https://multimedia.metrocuadrado.com/MC6287274/MC6287274_14_p.jpg', 'whatsappMessage': 'Hola, estoy interesado en el anuncio con ID: MC6287274 que encontré en Metrocuadrado. Apartamento en Arriendo, CHICO NORTE   Zona Urbana, Bogotá D.C. https://www.metrocuadrado.com', 'badge': None, 'areaPrivada': None, 'midinmueble': 'MC6287274', 'mtipoinmueble': {'id': '1', 'nombre': 'Apartamento'}, 'mtiponegocio': 'arriendo', 'mvalorventa': None, 'mvalorarriendo': 4200000, 'marea': 60.0, 'mareac': 60.0, 'areaprivada': None, 'mnrocuartos': '1', 'mnrobanos': '2', 'mnrogarajes': '1', 'mciudad': {'id': '1', 'nombre': 'Bogotá D.C.'}, 'ptags': None, 'mzona': {'id': '1', 'nombre': 'Norte'}, 'mbarrio': 'CR  GREEN HOUSE', 'mnombrecomunbarrio': 'CHICO NORT

,ID,Link,Titulo,Valor_Arriendo,Administracion,AreaConstruida,AreaHabitable,Habitaciones,Baños,Parqueaderos,Estrato,Barrio,Barrio_Comun,Latitud,Longitud,Antiguedad,Descripcion,Carac_Generales,Carac_Interiores,Carac_Exteriores
0,MC6287274,https://www.metrocuadrado.com/inmueble/arriend...,"Apartamento en Arriendo, CHICO NORTE Zona Ur...",4200000,570000,60.00,60.0,1,2,1,None,CR GREEN HOUSE,CHICO NORTE Zona Urbana,4.686728,-74.050500,None,None,,,
1,20347-M6098734,https://www.metrocuadrado.com/inmueble/arriend...,"Apartamento en Arriendo, La Cabrera, Bogotá D.C.",14000000,3095000,236.00,236.0,3,3,3,None,LA CABRERA,La Cabrera,4.673900,-74.042540,None,None,,,
2,MC2748942,https://www.metrocuadrado.com/inmueble/arriend...,"Apartamento en Arriendo, Cedritos, Bogotá D.C.",1600000,357050,54.00,54.0,1,1,1,None,LOS CEDRITOS,Cedritos,4.726174,-74.044110,None,None,,,
3,MC6317896,https://www.metrocuadrado.com/inmueble/arriend...,"Apartamento en Arriendo, LOS MONJES Engativá...",1350000,None,40.00,40.0,2,1,0,None,LOS MONJES,LOS MONJES Engativá,4.678188,-74.113690,None,None,,,
4,MC6332907,https://www.metrocuadrado.com/inmueble/arriend...,"Apartamento en Arriendo, REMANSO SUR Puente ...",1400000,40000,60.00,60.0,2,1,0,None,REMANSO SUR,REMANSO SUR Puente Aranda,4.602984,-74.120390,None,None,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,16285-M4917966,https://www.metrocuadrado.com/inmueble/arriend...,"Apartamento en Arriendo, LAS ROSALES, Bogotá D.C.",13500000,0,197.00,197.0,3,4,3,None,LAS ACACIAS,LAS ROSALES,4.650030,-74.051659,None,None,,,
61,MC6398809,https://www.metrocuadrado.com/inmueble/arriend...,"Apartamento en Arriendo, LA COLINA CAMPESTRE ...",2400000,None,29.48,34.0,1,1,1,None,EL PLAN,LA COLINA CAMPESTRE Colina y Alrededores,4.735467,-74.063820,None,None,,,
62,MC6411937,https://www.metrocuadrado.com/inmueble/arriend...,"Apartamento en Arriendo, COLINA EL PLAN Coli...",2000000,None,36.00,36.0,1,1,0,None,EL PLAN,COLINA EL PLAN Colina y Alrededores,4.740476,-74.067480,None,None,,,
63,MC6303471,https://www.metrocuadrado.com/inmueble/arriend...,"Apartamento en Arriendo, SANTA BARBARA CENTRAL...",5800000,None,70.00,100.0,2,2,2,None,SANTA BARBARA CENTRAL,SANTA BARBARA CENTRAL Santa Bárbara,4.698638,-74.037610,None,None,,,


In [69]:
{'contactPhone': '3166278996', 'whatsapp': '573166278996', 'title': 'Apartamento en Arriendo, COLINA EL PLAN   Colina y Alrededores, Bogotá D.C.', 'link': '/inmueble/arriendo-apartamento-bogota-el-plan-1-habitaciones-1-banos/MC6411937', 'imageLink': 'https://multimedia.metrocuadrado.com/MC6411937/MC6411937_2_p.jpg', 'whatsappMessage': 'Hola, estoy interesado en el anuncio con ID: MC6411937 que encontré en Metrocuadrado. Apartamento en Arriendo, COLINA EL PLAN   Colina y Alrededores, Bogotá D.C. https://www.metrocuadrado.com', 'badge': None, 'areaPrivada': None, 'midinmueble': 'MC6411937', 'mtipoinmueble': {'id': '1', 'nombre': 'Apartamento'}, 'mtiponegocio': 'arriendo', 'mvalorventa': None, 'mvalorarriendo': 2000000, 'marea': 36.0, 'mareac': 36.0, 'areaprivada': None, 'mnrocuartos': '1', 'mnrobanos': '1', 'mnrogarajes': '0', 'mciudad': {'id': '1', 'nombre': 'Bogotá D.C.'}, 'ptags': None, 'mzona': {'id': '3', 'nombre': 'Noroccidente'}, 'mbarrio': 'EL PLAN', 'mnombrecomunbarrio': 'COLINA EL PLAN   Colina y Alrededores', 'data': {'mprimerafotoinmueble': 'MC6411937_2', 'murldetalle': '/inmueble/arriendo-apartamento-bogota-el-plan-1-habitaciones-1-banos/MC6411937'}, 'mnombreconstructor': None, 'mnombreproyecto': None, 'midempresa': '', 'mcontactoinmobiliaria_fijo1': None, 'ubicacionaproximada': 'N', 'mestadoinmueble': 'Usado', 'mcontactosucursal_celular1': None, 'mgrupoid': '01', 'morden': '0.9352', 'moferente': 'Seller', 'signwall': None, 'haswhatsappbot': None, 'mgaleriainmueble': None, 'haswhatsappotp': None, 'cobertura': [], 'packages': None, 'tipovivienda': None, 'feria': None, 'distance': None, 'localizacion': {'lon': -74.06748, 'lat': 4.7404757}}


{'contactPhone': '3166278996',
 'whatsapp': '573166278996',
 'title': 'Apartamento en Arriendo, COLINA EL PLAN   Colina y Alrededores, Bogotá D.C.',
 'link': '/inmueble/arriendo-apartamento-bogota-el-plan-1-habitaciones-1-banos/MC6411937',
 'imageLink': 'https://multimedia.metrocuadrado.com/MC6411937/MC6411937_2_p.jpg',
 'whatsappMessage': 'Hola, estoy interesado en el anuncio con ID: MC6411937 que encontré en Metrocuadrado. Apartamento en Arriendo, COLINA EL PLAN   Colina y Alrededores, Bogotá D.C. https://www.metrocuadrado.com',
 'badge': None,
 'areaPrivada': None,
 'midinmueble': 'MC6411937',
 'mtipoinmueble': {'id': '1', 'nombre': 'Apartamento'},
 'mtiponegocio': 'arriendo',
 'mvalorventa': None,
 'mvalorarriendo': 2000000,
 'marea': 36.0,
 'mareac': 36.0,
 'areaprivada': None,
 'mnrocuartos': '1',
 'mnrobanos': '1',
 'mnrogarajes': '0',
 'mciudad': {'id': '1', 'nombre': 'Bogotá D.C.'},
 'ptags': None,
 'mzona': {'id': '3', 'nombre': 'Noroccidente'},
 'mbarrio': 'EL PLAN',
 'mnomb

In [71]:
import requests
import pandas as pd
import time
import re
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

def configurar_driver():
    options = Options()
    options.add_argument("--headless") # Ejecución en segundo plano para más velocidad
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    return driver

def completar_con_selenium(driver, url):
    """Extrae solo lo que la API de búsqueda no nos dio"""
    try:
        driver.get(url)
        time.sleep(4) # Tiempo reducido para optimizar
        html = driver.page_source
        
        # 1. Estrato
        estrato = re.search(r'Estrato\s*(\d+)', html)
        estrato = estrato.group(1) if estrato else "N/A"
        
        # 2. Antigüedad
        antiguedad = re.search(r'Antigüedad\s*([\d\w\s-]+)</div>', html)
        antiguedad = antiguedad.group(1) if antiguedad else "N/A"
        
        # 3. Descripción
        try:
            desc = driver.find_element(By.XPATH, "//h2[contains(text(), 'Acerca de este')]/following-sibling::div").text
        except:
            desc = "N/A"
            
        # 4. Características (Interiores/Exteriores)
        tags = driver.find_elements(By.TAG_NAME, "pt-tag")
        caracs = [t.get_attribute("element-id") for t in tags if t.get_attribute("element-id") and t.get_attribute("element-id") not in ["propertyState"]]
        
        return estrato, antiguedad, desc, ", ".join(list(set(caracs)))
    except:
        return "N/A", "N/A", "N/A", "N/A"

def scraping_pro_completo(ciudad="bogota", total=10):
    api_key = 'P1MfFHfQMOtL16Zpg36NcntJYCLFm8FqFfudnavl'
    headers = {
        'Content-type': 'application/json',
        'X-Api-Key': api_key,
        'User-Agent': 'Mozilla/5.0'
    }
    
    inmuebles_final = []
    driver = configurar_driver()
    
    # 1. Obtener base desde la API
    url_api = f"https://www.metrocuadrado.com/rest-search/search?realEstateTypeList=apartamento&realEstateBusinessList=arriendo&city={ciudad}&from=0&size={total}"
    
    print(f"Obteniendo {total} inmuebles desde la API...")
    r = requests.get(url_api, headers=headers)
    data = r.json()
    
    for i in data.get('results', []):
        link_completo = "https://www.metrocuadrado.com" + i.get('link', '')
        print(f"Completando datos de: {link_completo}")
        
        # 2. Combinar con datos de Selenium para lo que falta
        estrato, anti, desc, carac = completar_con_selenium(driver, link_completo)
        
        inmuebles_final.append({
            'ID': i.get('midinmueble'),
            'Link': link_completo,
            'Titulo': i.get('mtitle'),
            'Valor_Arriendo': i.get('mvalorarriendo'),
            'Administracion': i.get('mvaloradministracion'),
            'Area_m2': i.get('marea'),
            'Habitaciones': i.get('mnrohabitaciones'),
            'Baños': i.get('mnrobanos'),
            'Parqueaderos': i.get('mnrogarajes'),
            'Barrio': i.get('mnombrecomunbarrio'),
            'Latitud': i.get('mlatitud'),
            'Longitud': i.get('mlongitud'),
            # Datos extraídos por Selenium
            'Estrato': estrato,
            'Antiguedad': anti,
            'Descripcion': desc,
            'Todas_Caracteristicas': carac
        })
    
    driver.quit()
    return pd.DataFrame(inmuebles_final)

# Ejecución
df_final = scraping_pro_completo(total=50) # Prueba con 5 para ver la magia
df_final

Obteniendo 50 inmuebles desde la API...
Completando datos de: https://www.metrocuadrado.com/inmueble/arriendo-apartamento-bogota-7-de-agosto-1-habitaciones-1-banos/MC6380894
Completando datos de: https://www.metrocuadrado.com/inmueble/arriendo-apartamento-bogota-el-contador-3-habitaciones-4-banos-2-garajes/MC6409921
Completando datos de: https://www.metrocuadrado.com/inmueble/arriendo-apartamento-bogota-cr.-green-house-1-habitaciones-2-banos-1-garajes/MC6287274
Completando datos de: https://www.metrocuadrado.com/inmueble/arriendo-apartamento-bogota-la-cabrera-3-habitaciones-3-banos-3-garajes/20347-M6098734
Completando datos de: https://www.metrocuadrado.com/inmueble/arriendo-apartamento-bogota-los-cedritos-1-habitaciones-1-banos-1-garajes/MC2748942
Completando datos de: https://www.metrocuadrado.com/inmueble/arriendo-apartamento-bogota-los-monjes-2-habitaciones-1-banos/MC6317896
Completando datos de: https://www.metrocuadrado.com/inmueble/arriendo-apartamento-bogota-remanso-sur-2-habit

,ID,Link,Titulo,Valor_Arriendo,Administracion,Area_m2,Habitaciones,Baños,Parqueaderos,Barrio,Latitud,Longitud,Estrato,Antiguedad,Descripcion,Todas_Caracteristicas
0,MC6380894,https://www.metrocuadrado.com/inmueble/arriend...,None,1800000,None,50.0,None,1,0,SIETE DE AGOSTO Zona Urbana,None,None,3,N/A,"Apartamento de 60 metros, una habitación, luz ...","Cerca Supermercados, Número de piso 3, imageCo..."
1,MC6409921,https://www.metrocuadrado.com/inmueble/arriend...,None,4850000,None,160.0,None,4,2,CEDRITOS CONTADOR Zona Urbana,None,None,5,N/A,Amplio apartamento de ciento sesenta metros cu...,"Baño de Servicio, Citófonos, Estudio o bibliot..."
2,MC6287274,https://www.metrocuadrado.com/inmueble/arriend...,None,4200000,None,60.0,None,2,1,CHICO NORTE Zona Urbana,None,None,6,N/A,"Se arrienda apartamento en Burano, Calle 101 –...","Terraza Rooftop, Cerca Supermercados, Número d..."
3,20347-M6098734,https://www.metrocuadrado.com/inmueble/arriend...,None,14000000,None,236.0,None,3,3,La Cabrera,None,None,6,N/A,Descubre este exclusivo apartamento con 3 habi...,"Gimnasio, propertyType, Cerca Transporte Públi..."
4,MC2748942,https://www.metrocuadrado.com/inmueble/arriend...,None,1600000,None,54.0,None,1,1,Cedritos,None,None,4,N/A,"Códomo apartamento, Muy iluminado. Una alcoba ...","propertyType, Parqueadero visitantes, Cerca Pa..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,9951-M6004861,https://www.metrocuadrado.com/inmueble/arriend...,None,13000000,None,191.0,None,4,2,LOS ROSALES,None,None,N/A,N/A,N/A,N/A
61,MC6398809,https://www.metrocuadrado.com/inmueble/arriend...,None,2400000,None,34.0,None,1,1,LA COLINA CAMPESTRE Colina y Alrededores,None,None,N/A,N/A,N/A,N/A
62,MC6411937,https://www.metrocuadrado.com/inmueble/arriend...,None,2000000,None,36.0,None,1,0,COLINA EL PLAN Colina y Alrededores,None,None,N/A,N/A,N/A,N/A
63,MC6303471,https://www.metrocuadrado.com/inmueble/arriend...,None,5800000,None,100.0,None,2,2,SANTA BARBARA CENTRAL Santa Bárbara,None,None,N/A,N/A,N/A,N/A
